In [1]:
import datetime
from concordia.agents import entity_agent
from concordia.agents import entity_agent_with_logging 
from concordia.associative_memory import basic_associative_memory
from concordia.components import agent as agent_components
from concordia.components.agent import concat_act_component
from concordia.contrib import language_models as language_model_utils
from concordia.contrib.language_models.ollama import ollama_model
from concordia.environment.engines import asynchronous
from concordia.prefabs.simulation import generic as simulation
import concordia.prefabs.game_master as game_master_prefabs
import concordia.prefabs.entity as entity_prefabs
from concordia.typing import prefab as prefab_lib
from concordia.utils import async_measurements as async_measurements_lib
from concordia.utils import helper_functions
import pandas as pd
from IPython import display
import numpy as np
import sentence_transformers
import ollama
import json
import os
import yaml

In [2]:
os.environ["OLLAMA_HOST"] = "http://localhost:11434"

model = ollama_model.OllamaLanguageModel(
    model_name="deepseek-r1:8b",
)

In [3]:
with open('personas.yaml', 'r', encoding='utf-8') as f:
    yaml_data = yaml.safe_load(f)['personas']

In [4]:
class MpnetEmbedder:
    """Embedder using HuggingFace, adpted to Concordia."""
    
    def __init__(self, model):
        self._model = model
    def __call__(self, text: str) -> np.ndarray:
        return self._model.encode(text, show_progress_bar=False).astype(np.float32)
    def embed(self, text: str) -> np.ndarray:
        # Pega a string, encode, e garante que é um numpy array de float32
        return self._model.encode(text, show_progress_bar=False).astype(np.float32)

    def embed_sentence(self, text: str) -> np.ndarray:
        # Apenas um alias, pois algumas partes do Concordia preferem esse nome
        return self.embed(text)

In [5]:
st_model = sentence_transformers.SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
embedder = MpnetEmbedder(st_model)

In [8]:
def load_mapping():
    with open("mfq_mapping.json") as f:
        mapping = json.load(f)

    return mapping

In [9]:
def analyze_results(results, mapping):

    analysis = []

    for r in results:

        scores = compute_mfq_scores(
            r["answers"],
            mapping
        )

        analysis.append({
            "persona": r["persona"],
            **scores
        })

    return analysis

In [7]:
def compute_mfq_scores(answers, mapping):

    scores = {}

    for foundation, questions in mapping.items():

        values = [] 

        for q in questions:
            if q in answers:
                values.append(answers[q])

        scores[foundation] = np.mean(values)

    return scores

In [8]:
with open("results.json", "r", encoding="utf-8") as f:
    json_results = json.load(f)

mapping = load_mapping()

analysis = analyze_results(json_results, mapping)

df_mft = pd.DataFrame(analysis)

df_mft = df_mft.dropna(subset=['care', 'fairness', 'loyalty', 'authority', 'purity'])

NameError: name 'load_mapping' is not defined

In [9]:
def build_concordia_agent(persona_yaml, mft_scores):
    name = persona_yaml['name']
    
    interests = "\n- ".join(persona_yaml['interests'])
    beliefs = "\n- ".join(persona_yaml['beliefs'])
    
    # MUDANÇA 2: O Prompt agora força o agente a cuspir JSON purinho.
    identity_prompt = f"""
You are {name}, {persona_yaml['age_range']} years old, living in {persona_yaml['location']}.
Occupation: {persona_yaml['occupation']}
Description: {persona_yaml['small_description']}

Your main interests are:
- {interests}

Your core beliefs are:
- {beliefs}

YOUR MORAL FOUNDATIONS (Scale from 0 to 5, where 5 is the most important to you):
- Care/Harm: {mft_scores['care']:.2f}
- Fairness/Cheating: {mft_scores['fairness']:.2f}
- Loyalty/Betrayal: {mft_scores['loyalty']:.2f}
- Authority/Subversion: {mft_scores['authority']:.2f}
- Purity/Degradation: {mft_scores['purity']:.2f}

SOCIAL MEDIA BEHAVIOR INSTRUCTION:
When reading a news piece or post, internally analyze how it affects your moral foundations. 
If the news attacks a foundation you scored high on (above 4.0), you will feel outraged and likely reply with criticism or make an angry post.
If the news validates a foundation you scored high on, you will upvote it or reply with support.
If the news mostly involves foundations you scored low on (below 3.0), you will ignore it.

CRITICAL OUTPUT INSTRUCTION - YOU ARE AN API:
You are interacting directly with a digital forum backend. Every action you take MUST be a valid JSON object matching one of these exact formats. Do NOT output any markdown (like ```json), introduction, or conversational text outside the JSON.

Choose ONLY ONE action per turn from these options:
{{"action": "post", "author": "{name}", "title": "Your Title", "content": "Your thoughts"}}
{{"action": "reply", "author": "{name}", "post_id": "ID_OF_THE_POST", "content": "Your reply"}}
{{"action": "upvote", "author": "{name}", "post_id": "ID_OF_THE_POST"}}
{{"action": "downvote", "author": "{name}", "post_id": "ID_OF_THE_POST"}}
{{"action": "ignore", "author": "{name}"}}
"""

    raw_memory_list = []
    memory = agent_components.memory.ListMemory(memory_bank=raw_memory_list)
    
    identity = agent_components.constant.Constant(state=identity_prompt)
    
    act_comp = concat_act_component.ConcatActComponent(model=model)
    
    agent = entity_agent_with_logging.EntityAgentWithLogging(
        agent_name=name,
        act_component=act_comp, 
        context_components={    
            'memory': memory,
            'identity': identity
        },
        measurements=async_measurements_lib.ReactiveMeasurements()
    )
    return agent

In [13]:
def divide_into_batches(data_list, batch_size):
    for i in range(0, len(data_list), batch_size):
        yield data_list[i:i + batch_size]

persona_batches = list(divide_into_batches(yaml_data, 5))

for batch_index, yaml_batch in enumerate(persona_batches):
    print(f"\n{'='*50}\nStarting Batch {batch_index + 1} (ASYNC SOCIAL MEDIA)\n{'='*50}")
    
    active_agents = []
    
    for persona_yaml in yaml_batch:
        name = persona_yaml['name']
        mft_row = df_mft[df_mft['persona'] == name]
        
        if not mft_row.empty:
            mft_scores = mft_row.iloc[0]
            agent = build_concordia_agent(persona_yaml, mft_scores)
            active_agents.append(agent)
            print(f"Agent {name} successfully initialized.")

    
    gm_memory_bank = basic_associative_memory.AssociativeMemoryBank(
        sentence_embedder=embedder
    )

    # Inicializamos o seu Prefab avançado
    gm_prefab = game_master_prefabs.async_social_media.GameMaster( 
        entities=active_agents,
        params={
            'name': 'Forum System',
            'forum_name': 'Polítics and Daily Life Forum',
            'measurements': async_measurements_lib.ReactiveMeasurements()
        }
    )

    # Constrói o GM
    gm = gm_prefab.build(model=model, memory_bank=gm_memory_bank)

    sim_engine = asynchronous.Asynchronous()
    
    entities = active_agents + [gm]
    
    test_news = "SYSTEM BROADCAST POST: Activists invade productive farm claiming environmental damages and demand land redistribution."

    print(f"\n--- Iniciando a simulação assíncrona do Fórum ---")

    # 3. Dá a largada na simulação usando o run_loop!
    # Ele gerencia os turnos e as threads automaticamente
    sim_engine.run_loop(
        game_masters=[gm],           # Passamos o gm dentro de uma lista
        entities=active_agents,      # Passamos nossos 5 agentes
        premise=test_news,           # Injetamos a notícia inicial
        max_steps=4,                 # Duração da simulação (em vez do for loop)
        verbose=True                 # ATIVE O VERBOSE! Ele vai imprimir tudo o que acontece no terminal
    )
    
    recent_logs = gm.get_component('__memory__').retrieve_recent(limit=10)
    for log in recent_logs:
        if "action" in log.lower() or "post" in log.lower():
            print(f"[FEED FINAL] {log}")


Starting Batch 1 (ASYNC SOCIAL MEDIA)
Agent Roberto Almeida successfully initialized.
Agent Carlos Tavares successfully initialized.
Agent Ana Paula Santos successfully initialized.
Agent Irmã Maria Clara successfully initialized.
Agent Marcos Ferreira successfully initialized.

--- Iniciando a simulação assíncrona do Fórum ---
Terminate? No
Terminate? No
Terminate? No
Terminate? No
Terminate? No
Terminate? No
Entity Roberto Almeida observed: Polítics and Daily Life Forum: No new activity.
Entity Roberto Almeida is next to act. They must respond in the format: "ActionSpec(call_to_action='What does {name} do on the forum? Respond in JSON format with one of:\n{{"action": "post", "author": "{name}", "title": "...", "content": "..."}}\n{{"action": "reply", "author": "{name}", "post_id": "...", "content": "..."}}\n{{"action": "upvote", "author": "{name}", "post_id": "..."}}\n{{"action": "downvote", "author": "{name}", "post_id": "..."}}\n', output_type=<OutputType.FREE: 'free'>, options=()

In [ ]:
instances = [
    prefab_lib.InstanceConfig(
        prefab='basic__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Oliver Cromwell',
            'goal': 'become lord protector',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='basic__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'King Charles I',
            'goal': 'avoid execution for treason',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='generic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'default rules',
            # Comma-separated list of thought chain steps.
            'extra_event_resolution_steps': '',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='formative_memories_initializer__GameMaster',
        role=prefab_lib.Role.INITIALIZER,
        params={
            'name': 'initial setup rules',
            'next_game_master_name': 'default rules',
            'shared_memories': [
                'The king was captured by Parliamentary forces in 1646.',
                'Charles I was tried for treason and found guilty.',
            ],
        },
    ),
]

In [ ]:
import inspect
assinatura = inspect.signature(asynchronous.Asynchronous.__init__)
print(f"O Motor Assíncrono exige estes parâmetros: {assinatura}")